# Reconstruction Comparison

This notebook compares original and reconstructed images produced by the CAE or GAN-AE experiments.

## Setup

In [ ]:
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr


## Helper Functions

In [ ]:
def load_image(path, size=None):
    image = Image.open(path).convert("RGB")
    if size is not None:
        image = image.resize(size)
    return np.asarray(image) / 255.0

def show_pair(original, reconstructed, title="Original vs Reconstructed"):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(reconstructed)
    axes[1].set_title("Reconstructed")
    axes[1].axis("off")

    fig.suptitle(title)
    plt.show()

def compute_metrics(original, reconstructed):
    original = np.clip(original, 0, 1)
    reconstructed = np.clip(reconstructed, 0, 1)

    psnr_value = psnr(original, reconstructed, data_range=1.0)
    ssim_value = ssim(original, reconstructed, channel_axis=-1, data_range=1.0)

    return {
        "PSNR": psnr_value,
        "SSIM": ssim_value
    }


## Set Image Paths

In [ ]:
# Update these paths based on your experiment outputs.
ORIGINAL_IMAGE = Path("../data/celeba/img_align_celeba/example_original.jpg")
RECONSTRUCTED_IMAGE = Path("../runs/celeba_cae/example_reconstructed.jpg")

# Replace the paths above with your own image files.
print("Original path:", ORIGINAL_IMAGE)
print("Reconstructed path:", RECONSTRUCTED_IMAGE)


## Load and Compare Images

In [ ]:
if not ORIGINAL_IMAGE.exists():
    raise FileNotFoundError(f"Original image not found: {ORIGINAL_IMAGE}")

if not RECONSTRUCTED_IMAGE.exists():
    raise FileNotFoundError(f"Reconstructed image not found: {RECONSTRUCTED_IMAGE}")

original = load_image(ORIGINAL_IMAGE, size=(64, 64))
reconstructed = load_image(RECONSTRUCTED_IMAGE, size=(64, 64))

show_pair(original, reconstructed)
metrics = compute_metrics(original, reconstructed)
metrics


## Batch Comparison from Folders

In [ ]:
# Optional: compare matching images from two folders.
ORIGINAL_DIR = Path("../data/original_samples")
RECON_DIR = Path("../runs/reconstructed_samples")

if ORIGINAL_DIR.exists() and RECON_DIR.exists():
    original_files = sorted([p for p in ORIGINAL_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    recon_files = sorted([p for p in RECON_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])

    results = []
    for orig_path, recon_path in zip(original_files, recon_files):
        orig = load_image(orig_path, size=(64, 64))
        recon = load_image(recon_path, size=(64, 64))
        m = compute_metrics(orig, recon)
        m["original"] = orig_path.name
        m["reconstructed"] = recon_path.name
        results.append(m)

    print(f"Compared {len(results)} image pairs.")
    results[:5]
else:
    print("Batch folders not found. Update ORIGINAL_DIR and RECON_DIR if needed.")
